In [1]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

c:\Users\swapn\OneDrive\Desktop\new_repo\langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_docs=[]
pdf_path = Path("C:\\Users\\swapn\\OneDrive\\Desktop\\new_repo\\langchain\\docs").glob("*.pdf")
for path in pdf_path:
    loader = PyPDFLoader(path)
    docs = loader.load()
    all_docs.extend(docs)

print(f"read {len(all_docs)} documents")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 63 0 (offset 0)
Ignoring wrong 

read 150 documents


In [3]:
print(type(all_docs[0]))

<class 'langchain_core.documents.base.Document'>


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
chunks=splitter.split_documents(all_docs)

In [6]:
print(chunks[0].page_content)

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

In [9]:
embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4693.09it/s]


In [10]:
import os
from langchain_community.vectorstores import Chroma

In [11]:
DB_DIR = "./chroma_db"

if os.path.exists(DB_DIR):

    print("Loading existing vector DB...")

    vectorstore = Chroma(
        persist_directory=DB_DIR,
        embedding_function=embedding_model
    )

else:

    print("Creating new vector DB...")

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=DB_DIR
    )

Creating new vector DB...


In [12]:
retriever = vectorstore.as_retriever(
    search_type="mmr", #maximum marginal relevance, it does it based on relevance and diversity of the results
 
    search_kwargs={
        "k": 3,
        "fetch_k": 6
    }
)

In [13]:
query = "numpy"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\nRESULT {i+1}")
    print(doc.page_content[:500])


RESULT 1
Python For Data Analysis Lab Subcode: B24EH0406 
School of Computer Science and Engineering Page 21 
 
 
 
Viva Questions and Answers 
What is NumPy in Python? 
NumPy (Numerical Python) is a Python library used for fast mathematical computations. It provides support for 
multidimensional arrays and a wide range of mathematical, statistical, and linear algebra functions.  
 
2. What is the difference between a Python list and a NumPy array?

RESULT 2
• Used aggregation functions like sum(), mean(), min(), max(). 
. 
Code 
(a) Creating Arrays and Basic Arithmetic Operations 
import numpy as np 
 
# Creating arrays 
a = np.array([1, 2, 3, 4, 5]) 
b = np.array([10, 20, 30, 40, 50]) 
 
print("Array a:", a) 
print("Array b:", b) 
 
# Arithmetic operations 
print("Addition:", a + b) 
print("Subtraction:", b - a) 
print("Multiplication:", a * b) 
print("Division:", b / a) 
 
(b) Array Slicing and Reshaping 
# Slicing

RESULT 3
6. Why is NumPy faster than lists? 
Because NumPy uses co

In [14]:
from langchain_ollama import ChatOllama

In [15]:
llm = ChatOllama(model="llama3:8b",temperature=0.1)

In [16]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are a document question-answering system.

Rules:
- Be concise.
- Do not greet the user.
- Do not say "I'm happy to help".
- Answer ONLY from the provided context.
- If answer is unavailable, say:
  "I could not find the answer in the documents."

Context:
{context}

Question:
{question}
""")

In [17]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

In [ ]:
chat_history = []

while True:

    query = input("\nAsk Question: ")

    if query.lower() == "exit":
        break
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join([
        f"""
    SOURCE: {doc.metadata.get("source")}
    PAGE: {doc.metadata.get("page")}

    CONTENT:
    {doc.page_content}
    """
        for doc in retrieved_docs
    ])
    response = chain.invoke({
        "history": "\n".join(chat_history),
        "context": context,
        "question": query
    })
    print("\nRETRIEVED CHUNKS:\n")

    for i, doc in enumerate(retrieved_docs):
        print(f"\n--- CHUNK {i+1} ---")
        print(doc.page_content[:300])
    print("\nANSWER:")
    print(response)
    chat_history.append(f"User: {query}")
    chat_history.append(f"Assistant: {response}")

    print("\nSOURCES:")

    for i, doc in enumerate(retrieved_docs):

        source = doc.metadata.get("source", "Unknown")

        page = doc.metadata.get("page", "Unknown")

        print(f"{i+1}. {source} | Page {page}")